In [22]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [11]:
load_dotenv()
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

In [8]:
doc_loader = PyPDFLoader('../book.pdf')

docs = doc_loader.load()

In [10]:
vector_store = Chroma.from_documents(
    documents=docs,
    embedding=OpenAIEmbeddings(model='text-embedding-3-small')
)

In [31]:
base_retriever = vector_store.as_retriever(
    search_type = 'similarity',
    search_kwargs = {'k': 5}
)

mqr_retriever = MultiQueryRetriever.from_llm(
    retriever=vector_store.as_retriever(search_kwargs = {'k': 5}),
    llm=llm
)

In [44]:
query = "How to design efficient and scalable systems?"

base_ret_results = base_retriever.invoke(query)

In [45]:
mqr_results = mqr_retriever.invoke(query)

In [39]:
context_by_base_ret = "\n\n".join(doc.page_content for doc in base_ret_results)
context_by_mqr = "\n\n".join(doc.page_content for doc in mqr_results)

In [40]:
prompt = PromptTemplate(
    template = '''You are a helpful AI assistant.

    Answer the question using ONLY the provided context.
    If the answer is not available in the context, say so.

    Context:
    {context}

    Question:
    {question}''',

    input_variables=['context', 'question']
)

In [43]:
parser = StrOutputParser()

chain = prompt | llm | parser

base_ret_llm_results = chain.invoke({
    'context': context_by_base_ret,
    'question': query
})

mqr_ret_llm_results = chain.invoke({
    'context': context_by_mqr,
    'question': query
})

print(base_ret_llm_results)
print("\n", "MQR retriever: \n", mqr_ret_llm_results)

To design efficient and scalable systems, you should follow these techniques:

1. Keep the web tier stateless.
2. Build redundancy at every tier.
3. Cache data as much as you can.
4. Support multiple data centers.
5. Host static assets in a CDN.
6. Scale your data tier by sharding.
7. Split tiers into individual services.
8. Monitor your system and use automation tools.

These strategies provide a good foundation for tackling the challenges of scaling systems to support millions of users.

 MQR retriever: 
 To design efficient and scalable systems, consider the following strategies:

1. **Keep web tier stateless**: This allows for easier scaling and management.
2. **Build redundancy at every tier**: This helps prevent single points of failure.
3. **Cache data as much as you can**: Caching improves performance by reducing load on databases.
4. **Support multiple data centers**: This enhances availability and reduces latency for users in different regions.
5. **Host static assets in CDN*